# Exploratory statistical analysis

This study is exploratory. We did not perform a sample-size determination or power analysis based on an expected effect size. The statistical tests below are not confirmatory and should be interpreted descriptively.

In [1]:
from scipy import stats
import numpy as np
import json
import math

In [2]:
list_layers = ["encoder", "latent", "decoder", "all"]

list_rois = [
    "all",
    "early",
    "midventral",
    "ventral",
    "midlateral",
    "lateral",
    "midparietal",
    "parietal",
]

dict_models = {
    "Standard VAE": "/workspace/data/rsa/vanilla_from_ae_beta_vae_loss_standard_beta0.001_recon_loss_l2/cocoDoerig/run_0/rsa_rdm_{layer}_regression_special515.json",
    "LLM-guided VAE": "/workspace/data/rsa/beta_vae_loss_llm_alignment_beta0.001_recon_loss_l2_llm_alignment_loss_cosine_similarity_gamma0.5/cocoDoerig/run_0/rsa_rdm_{layer}_regression_special515.json",
    "Standard AE": "/workspace/data/rsa/vanilla_ae_loss_l2/cocoDoerig/run_0/rsa_rdm_{layer}_regression_special515.json",
    "LLM-guided AE": "/workspace/data/rsa/ae_loss_l2/cocoDoerig/run_0/rsa_rdm_{layer}_regression_special515.json",
}

# architecture pairs: (standard model, llm-guided model)
pairs = {
    "VAE": ("Standard VAE", "LLM-guided VAE"),
    "AE":  ("Standard AE",  "LLM-guided AE"),
}

# H1: LLM-guided > Standard
Controlling false discovery late with Benjamini-Hochberg (FDR=0.05).



In [3]:
# load: full_data[layer][model_name][roi] = list of 8 participant RSA scores
full_data = {}
for layer in list_layers:
    full_data[layer] = {}
    for model_name, path_template in dict_models.items():
        with open(path_template.format(layer=layer), "r") as f:
            full_data[layer][model_name] = json.load(f)

In [9]:
test_rois = [r for r in list_rois if r != "all"] # remove all region
# 1) collect every test (paired t-test, H1: LLM-guided > Standard)
records = []
for layer in list_layers:
    for architecture, (std_model, llm_model) in pairs.items():
        for roi in test_rois:
            stdr_vals = np.array(full_data[layer][std_model][roi])
            llm_vals = np.array(full_data[layer][llm_model][roi])
            t_stat, p_val = stats.ttest_rel(llm_vals, stdr_vals, alternative="greater")
            records.append({
                "layer": layer, "arch": architecture, "roi": roi,
                "mean_stdr": stdr_vals.mean(), "mean_llm": llm_vals.mean(),
                "t": t_stat, "p": p_val,
            })

# 2) BH-FDR correction across the whole
alpha = 0.05
q_vals = stats.false_discovery_control([r["p"] for r in records])
for r, q in zip(records, q_vals):
    r["p_fdr"] = q
    r["sig"] = q < alpha

def ceil_p(p, decimals=3):
    factor = 10 ** decimals
    return math.ceil(p * factor) / factor


# 3) report
print(f"Benjamini–Hochberg FDR control, alpha={alpha}, "
      f"family size = {len(records)} tests\n")
for layer in list_layers:
    for arch in pairs:
        print(f"=== layer={layer} | {arch} (LLM-guided vs Standard) ===")
        #print(f"{'ROI':<12} {'mean_stdr':>9} {'mean_llm':>9} {'t':>8} {'p':>10} {'p_fdr':>10}")
        print(f"{'ROI':<12} {'p':>10} {'p_fdr':>10}")
        for r in records:
            if r["layer"] == layer and r["arch"] == arch:
                star = "*" if r["sig"] else ""
                # print(f"{r['roi']:<12} {r['mean_stdr']:9.3f} {r['mean_llm']:9.3f} "
                #       f"{r['t']:8.3f} {r['p']:10.3f} {r['p_fdr']:10.3f} {star}")
                print(f"{r['roi']:<12} {r['p']:10.3f} {r['p_fdr']:10.3f} {star}")
        print()


Benjamini–Hochberg FDR control, alpha=0.05, family size = 56 tests

=== layer=encoder | VAE (LLM-guided vs Standard) ===
ROI                   p      p_fdr
early             0.936      1.000 
midventral        0.980      1.000 
ventral           1.000      1.000 
midlateral        0.337      0.591 
lateral           0.002      0.005 *
midparietal       0.947      1.000 
parietal          0.976      1.000 

=== layer=encoder | AE (LLM-guided vs Standard) ===
ROI                   p      p_fdr
early             1.000      1.000 
midventral        0.914      1.000 
ventral           0.975      1.000 
midlateral        0.833      1.000 
lateral           0.032      0.061 
midparietal       0.856      1.000 
parietal          0.998      1.000 

=== layer=latent | VAE (LLM-guided vs Standard) ===
ROI                   p      p_fdr
early             0.176      0.318 
midventral        0.000      0.001 *
ventral           0.000      0.000 *
midlateral        0.000      0.001 *
lateral         